# PT-04 — Group Relative Policy Optimization (GRPO)

**Serie** : Post-Training SOTA 2024-2025 (Epic #1742, sub-issue #1757)
**LIVRABLE CLE** de l'Epic #1742
**Pre-requis** : PT-01 (intro), PT-02 (SFT), PT-03 (DPO) recommandes
**Objectifs pedagogiques** :
1. Comprendre pourquoi GRPO est la technique cle derriere Deepseek-R1 (janvier 2025)
2. Maitriser la formule de l'avantage relatif intra-group (normalisation sans critic)
3. Implementer `trl.GRPOTrainer` avec une reward function personnalisable
4. Explorer l'impact du group size sur la variance et la memoire
5. Evaluer qualitativement les outputs avant/apres GRPO sur des prompts reasoning

**References cles** :
- Shao et al., "DeepSeekMath: Pushing the Limits of Mathematical Reasoning in Open Language Models" (2024) — paper origin GRPO
- Deepseek-AI, "DeepSeek-R1: Incentivizing Reasoning Capability in LLMs via Reinforcement Learning" (2025)
- HuggingFace TRL GRPOTrainer integration docs (2025)

## 1. Pourquoi GRPO a revolutionne le post-training (janvier 2025)

### Le choc Deepseek-R1

En janvier 2025, Deepseek-AI publie Deepseek-R1, un modele de raisonnement qui rivalise avec OpenAI o1 **sans donnees de raisonnement humain**. La cle : un entrainement RL pure avec GRPO, qui a fait emerger des comportements de "chain-of-thought" spontanes.

**Timeline** :

| Date | Evenement | Impact |
|------|-----------|--------|
| 2024-02 | DeepSeekMath (GRPO origin) | GRPO pour raisonnement mathematique |
| 2024-12 | Deepseek-V3 (MTP, MoE) | Architecture de base |
| 2025-01 | **Deepseek-R1** | GRPO → reasoning emergent sans SFT |
| 2025-02 | R1-Distill (1.5B→70B) | Modeles distilles a partir de R1 |
| 2025-Q1 | GRPO → TRL integration | Democratise via HuggingFace |

### Ce que GRPO a de unique

**Pas de critic network.** En RL classique (PPO), on entraine un reseau separé (critic/value function) pour estimer l'avantage. GRPO elimine completement cette composante :

| Composant | PPO classique | GRPO |
|-----------|---------------|------|
| Policy network | Oui | Oui |
| Reference model | Oui (KL penalty) | Oui (KL via beta) |
| Critic/Value function | **Oui** | **NON** |
| Reward model | Oui (appris) | Oui ou fonction exacte |
| Memoire totale | ~4x le modele | ~2x le modele |

**Reduction memoire : ~50%** vs PPO. C'est ce qui rend GRPO praticable sur GPU 8 Go.

## 2. Mathematiques du GRPO — l'avantage relatif intra-group

### L'idee centrale

Pour un prompt $x$, GRPO echantillonne **G completions** depuis la policy : $\{y_1, y_2, ..., y_G\}$.

Chaque completion recoit un reward $R_i$ (via reward model ou fonction exacte).

L'avantage est calcule par **normalisation intra-group** :

$$\hat{A}_i = \frac{R_i - \mu_G}{\sigma_G}$$

ou $\mu_G = \frac{1}{G}\sum_{j=1}^{G} R_j$ et $\sigma_G = \sqrt{\frac{1}{G}\sum_{j=1}^{G}(R_j - \mu_G)^2}$

### Le loss GRPO

$$\mathcal{L}_{\text{GRPO}}(\theta) = -\mathbb{E}_{x, \{y_i\}_{i=1}^{G}} \left[ \frac{1}{G} \sum_{i=1}^{G} \hat{A}_i \cdot \min\left( \rho_i \cdot r_i(\theta), \text{clip}(\rho_i, 1-\epsilon, 1+\epsilon) \cdot r_i(\theta) \right) - \beta \cdot D_{\text{KL}}(\pi_\theta || \pi_{\text{ref}}) \right]$$

**Decomposition** :

- $r_i(\theta) = \frac{\pi_\theta(y_i | x)}{\pi_{\theta_{\text{old}}}(y_i | x)}$ : ratio de probabilite (comme PPO)
- $\rho_i$ : rapport de log-probabilite
- $\epsilon$ : clipping parameter (typique 0.2)
- $\beta$ : coefficient KL (typique 0.04)
- $G$ : group size (typique 4-16)

### Pourquoi la normalisation intra-group fonctionne

1. **Pas besoin de baseline apprise** : la moyenne du groupe sert de baseline naturelle
2. **Variance reduite** : normaliser par $\sigma_G$ reduit le bruit
3. **Gradients informatifs** : meme avec des rewards sparse, le classement relatif fournit un signal
4. **Scale avec le compute** : plus de G = meilleure estimation, mais plus de memoire

### Intuition geometric

Imaginez un prompt "Solve : $2x + 3 = 7$". GRPO genere G=4 reponses :
- $y_1$ : "x = 2" → reward 1.0 (correct)
- $y_2$ : "x = 5" → reward 0.0 (incorrect)
- $y_3$ : "x = 2, verification : 2*2+3=7" → reward 1.0 (correct + raisonnement)
- $y_4$ : "je ne sais pas" → reward 0.0 (refusal)

Avantages : $\hat{A} = [0.41, -0.82, 0.41, -0.82]$. Le modele apprend a preferer $y_1$ et $y_3$, eviter $y_2$ et $y_4$.

## 3. Pseudocode d'une etape GRPO

```
pour chaque batch de prompts:
    # 1. Echantillonner G completions par prompt
    pour chaque prompt x:
        completions = [sample(pi_theta, x) for _ in range(G)]

    # 2. Calculer les rewards
    rewards = [reward_function(y) for y in completions]

    # 3. Normaliser intra-group (l'avantage GRPO)
    mu = mean(rewards)
    sigma = std(rewards)
    advantages = [(r - mu) / (sigma + eps) for r in rewards]

    # 4. Clipped surrogate objective (comme PPO)
    ratios = [pi_theta(y|x) / pi_old(y|x) for y in completions]
    clipped = clip(ratios, 1-epsilon, 1+epsilon)
    policy_loss = -min(ratio * advantage, clipped * advantage)

    # 5. KL penalty
    kl = D_KL(pi_theta || pi_ref)

    # 6. Update
    total_loss = policy_loss + beta * kl
    total_loss.backward()
    optimizer.step()
```

**Complexite** : $O(B \times G \times L)$ ou B=batch size, G=group size, L=sequence length.
**Memoire** : $O(G \times L)$ par prompt en plus du modele (stockage des G log-probs).

## 4. Verification de l'environnement

GRPO echantillonne G completions par prompt, ce qui consomme plus de VRAM que DPO. Avec 4-bit + LoRA + group_size=4, on reste sous 6 Go.

In [1]:
import sys
import os
import platform

print(f"Python : {sys.version}")
print(f"Plateforme : {platform.platform()}")

LOAD_MODEL_AND_TRAIN = False
print(f"\nLOAD_MODEL_AND_TRAIN = {LOAD_MODEL_AND_TRAIN}")
if not LOAD_MODEL_AND_TRAIN:
    print("(Mode CPU-safe : generation + training seront skippees)")

Python : 3.13.7 (tags/v3.13.7:bcee1c3, Aug 14 2025, 14:15:11) [MSC v.1944 64 bit (AMD64)]
Plateforme : Windows-11-10.0.26200-SP0

LOAD_MODEL_AND_TRAIN = False
(Mode CPU-safe : generation + training seront skippees)


In [2]:
import torch

CUDA_AVAILABLE = torch.cuda.is_available()
if CUDA_AVAILABLE:
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"CUDA disponible : {gpu_name} ({gpu_mem:.1f} Go)")
else:
    print("CUDA non disponible (CPU uniquement)")

CUDA disponible : NVIDIA GeForce RTX 3070 Laptop GPU (8.0 Go)


## 5. Design de la reward function

Contrairement a DPO (qui utilise des paires pre-labelisees), GRPO utilise une **reward function** callable qui prend une completion et retourne un float.

Pour ce notebook pedagogique, on definit des reward functions simples :

| Reward | Logique | Quand l'utiliser |
|--------|---------|------------------|
| `length_reward` | Penalise les reponses trop courtes ou trop longues | Alignement format |
| `keyword_reward` | +1 si le mot-cle attendu est present | QA simple |
| `json_format_reward` | +1 si JSON valide avec cle attendue | Generation structuree |
| `combined_reward` | Moyenne ponderee des 3 | Usage general |

In [3]:
import json
import re

def length_reward(completion: str, target_min: int = 20, target_max: int = 200) -> float:
    # Recompense basee sur la longueur. Penalise trop court/trop long.
    length = len(completion.strip())
    if length < target_min:
        return length / target_min * 0.5
    elif length > target_max:
        return max(0.0, 1.0 - (length - target_max) / target_max)
    else:
        return 1.0

def keyword_reward(completion: str, keywords: list = None) -> float:
    # Recompense si des mots-cles attendus sont presents.
    if keywords is None:
        keywords = ["therefore", "because", "since", "thus", "hence"]
    completion_lower = completion.lower()
    found = sum(1 for kw in keywords if kw in completion_lower)
    return found / len(keywords)

def json_format_reward(completion: str, required_key: str = "answer") -> float:
    # Recompense si la completion est un JSON valide avec cle attendue.
    try:
        parsed = json.loads(completion.strip())
        if isinstance(parsed, dict) and required_key in parsed:
            return 1.0
        elif isinstance(parsed, dict):
            return 0.5
        else:
            return 0.25
    except (json.JSONDecodeError, TypeError):
        return 0.0

def combined_reward(completion: str) -> float:
    # Reward combine : longueur + mots-cles + format.
    r_len = length_reward(completion)
    r_kw = keyword_reward(completion)
    return 0.4 * r_len + 0.6 * r_kw

# Tests unitaires des reward functions
print("Tests des reward functions :")
print(f"  length_reward('short') = {length_reward('short'):.2f}")
print(f"  length_reward('a' * 100) = {length_reward('a' * 100):.2f}")
print(f"  keyword_reward('Therefore, x = 2 because...') = {keyword_reward('Therefore, x = 2 because...'):.2f}")
json_test = '{"answer": 42}'
print(f"  json_format_reward('{json_test}') = {json_format_reward(json_test):.2f}")
print(f"  json_format_reward('not json') = {json_format_reward('not json'):.2f}")
print(f"  combined_reward('Therefore x=2 because the equation holds') = {combined_reward('Therefore x=2 because the equation holds'):.2f}")
print("\nReward functions pretes.")

Tests des reward functions :
  length_reward('short') = 0.12
  length_reward('a' * 100) = 1.00
  keyword_reward('Therefore, x = 2 because...') = 0.40
  json_format_reward('{"answer": 42}') = 1.00
  json_format_reward('not json') = 0.00
  combined_reward('Therefore x=2 because the equation holds') = 0.64

Reward functions pretes.


### Exercice 1 : Implementer une reward function de coherence logique

Les reward functions existantes (longueur, mots-cles, JSON) evaluent le format mais pas le fond. L'objectif est d'implementer une reward function `coherence_reward` qui detecte la presence de structures logiques dans la reponse (sillogismes, cause-effet, enumeration structuree).

**Objectif** : ecrire une fonction qui analyse une completion et retourne un score de coherence base sur la presence de connecteurs logiques, de numerotation structuree, et de longueur adequat.

**Indices** :
- # Etape 1 : Definir des patterns de connecteurs logiques ("donc", "par consequent", "premierement", "en conclusion")
- # Etape 2 : Compter le nombre de patterns trouves et normaliser par la longueur du texte
- # Indice : combiner avec un bonus pour les reponses de longueur intermediaire (ni trop courtes, ni trop longues)

In [1]:
def coherence_reward(completion: str) -> float:
    # TODO etudiant : implementer la reward de coherence logique
    
    # Etape 1 : definir les connecteurs logiques
    connecteurs = []  # TODO etudiant : liste de mots/phrases logiques
    
    # Etape 2 : compter les occurrences et normaliser
    completion_lower = completion.lower()
    score = 0.0  # TODO etudiant : calculer le score
    
    return score

print("Exercice a completer : coherence_reward")

Exercice a completer


## 6. Configuration LoRA + GRPO

**Contrainte memoire** : GRPO stocke G=4 sequences par prompt en plus du modele.
- Modele base 4-bit : ~0.4 Go
- LoRA adapters : ~15 Mo
- G=4 completions (max 512 tokens) : ~1.5 Go temporaire
- Total estime : ~3 Go (bien sous 8 Go RTX 3070)

In [4]:
from peft import LoraConfig, TaskType

LORA_CONFIG = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

print("LoRA config GRPO :")
print(f"  rank={LORA_CONFIG.r}, alpha={LORA_CONFIG.lora_alpha}")
print(f"  targets={LORA_CONFIG.target_modules}")

LoRA config GRPO :
  rank=8, alpha=16
  targets={'up_proj', 'q_proj', 'v_proj', 'gate_proj', 'down_proj', 'k_proj', 'o_proj'}


## 7. Configuration GRPO

Hyperparametres cles :
- **group_size = 4** : 4 completions par prompt (compromis variance/memoire)
- **beta = 0.04** : KL regularization (defaut Deepseek, plus faible que DPO 0.1)
- **clipping epsilon = 0.2** : meme que PPO
- **learning_rate = 1e-5** : entre SFT (2e-4) et DPO (5e-7)

In [5]:
GRPO_CONFIG_DICT = {
    "num_generations": 4,           # Group size G
    "beta": 0.04,                   # KL regularization (Deepseek default)
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 8,
    "learning_rate": 1e-5,
    "lr_scheduler_type": "cosine",
    "warmup_ratio": 0.1,
    "max_completion_length": 256,   # Short completions for demo
    "max_prompt_length": 128,
    "logging_steps": 2,
    "save_strategy": "no",
    "output_dir": "./grpo_output",
    "seed": 42,
    "bf16": True,
}

print("Configuration GRPO :")
for k, v in GRPO_CONFIG_DICT.items():
    print(f"  {k} = {v}")

Configuration GRPO :
  num_generations = 4
  beta = 0.04
  per_device_train_batch_size = 1
  gradient_accumulation_steps = 8
  learning_rate = 1e-05
  lr_scheduler_type = cosine
  warmup_ratio = 0.1
  max_completion_length = 256
  max_prompt_length = 128
  logging_steps = 2
  save_strategy = no
  output_dir = ./grpo_output
  seed = 42
  bf16 = True


### Exercice 2 : Analyser l'impact du group size sur la memoire

Le parametre `num_generations` (group size G) controle le nombre de completions generees par prompt. L'objectif est d'implementer une fonction `estimer_memoire_grpo` qui calcule la memoire supplementaire necessaire en fonction de G.

**Objectif** : ecrire une fonction qui estime la VRAM consommee par GRPO selon la taille du modele, la longueur des completions et le group size G.

**Indices** :
- # Etape 1 : Calculer la memoire de base du modele (params * bytes) + les activations
- # Etape 2 : Ajouter G * max_completion_length * hidden_size * 2 bytes (log-probs stockees)
- # Indice : comparer G=2, 4, 8, 16 et verifier lesquels tiennent sur 8 Go

In [1]:
def estimer_memoire_grpo(
    model_params_millions: float,
    group_size: int = 4,
    max_completion_length: int = 256,
    hidden_size: int = 896,
    quantization_bits: int = 4,
) -> dict:
    # TODO etudiant : estimer la VRAM pour GRPO
    
    # Etape 1 : memoire de base (modele quantize)
    bytes_per_param = quantization_bits / 8
    base_vram = None  # TODO etudiant : params * bytes / 1e9
    
    # Etape 2 : memoire pour les G completions (log-probs + activations)
    gen_vram = None  # TODO etudiant : G * length * hidden * 2 bytes / 1e9
    
    total = None  # TODO etudiant : base + gen + overhead
    fits_8gb = None  # TODO etudiant : total <= 8.0
    
    return {"total_gb": total, "fits_8gb": fits_8gb, "group_size": group_size}

print("Exercice a completer : estimation memoire GRPO")

Exercice a completer


## 8. Dataset de prompts pour GRPO

Le GRPO a besoin d'un dataset de **prompts uniquement** (pas de paires). Les completions sont generees par le modele pendant l'entrainement.

Pour le demo : un petit dataset de prompts "reasoning" (questions mathematiques/logiques simples).

In [6]:
# Dataset de prompts pour GRPO (demo)
prompts = [
    "What is 15 + 27? Think step by step.",
    "If a train travels 60 km/h for 2.5 hours, how far does it go?",
    "Is 97 a prime number? Explain your reasoning.",
    "What is the area of a circle with radius 5? Use pi = 3.14.",
    "Solve for x: 3x - 7 = 14.",
    "How many ways can you arrange 4 books on a shelf?",
    "What is 20% of 350?",
    "If it takes 5 machines 5 minutes to make 5 widgets, how long for 100 machines to make 100 widgets?",
    "What is the next prime after 50?",
    "Convert 72 degrees Fahrenheit to Celsius. Formula: C = (F-32) * 5/9.",
]

# Format conversation pour le chat template
from datasets import Dataset

def format_prompts(prompts_list):
    formatted = []
    for p in prompts_list:
        formatted.append({"prompt": [{"role": "user", "content": p}]})
    return formatted

dataset_grpo = Dataset.from_list(format_prompts(prompts))

print(f"Dataset GRPO : {len(dataset_grpo)} prompts")
print(f"Exemple : {dataset_grpo[0]['prompt']}")

Dataset GRPO : 10 prompts
Exemple : [{'role': 'user', 'content': 'What is 15 + 27? Think step by step.'}]


## 9. Construction du GRPOTrainer

`trl.GRPOTrainer` orchestre :
1. Generation de G completions par prompt
2. Evaluation des completions via la reward function
3. Calcul de l'avantage intra-group (normalisation)
4. Mise a jour de la policy avec clipped surrogate + KL penalty

In [7]:
if LOAD_MODEL_AND_TRAIN and CUDA_AVAILABLE:
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    from trl import GRPOTrainer, GRPOConfig
    from peft import get_peft_model

    MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

    # Quantization 4-bit
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

    print("Chargement du modele (4-bit)...")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
    )
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    print("Application LoRA...")
    model = get_peft_model(model, LORA_CONFIG)
    model.print_trainable_parameters()

    print("Configuration GRPOConfig...")
    grpo_config = GRPOConfig(**GRPO_CONFIG_DICT)

    print("Construction GRPOTrainer...")
    trainer = GRPOTrainer(
        model=model,
        args=grpo_config,
        processing_class=tokenizer,
        train_dataset=dataset_grpo,
        reward_funcs=[combined_reward],  # Pass reward function directly
    )

    print(f"\nGRPOTrainer pret. {len(dataset_grpo)} prompts, G={GRPO_CONFIG_DICT['num_generations']}.")
    print("Lancement de l'entrainement...")
    train_result = trainer.train()
    print(f"\nEntrainement GRPO termine.")
    print(f"Loss finale : {train_result.training_loss:.4f}")

else:
    print("Skip : entrainement GRPO non execute (LOAD_MODEL_AND_TRAIN=False ou pas de CUDA)")
    print()
    print("Pour executer :")
    print("  1. LOAD_MODEL_AND_TRAIN = True")
    print("  2. GPU avec >= 6 Go VRAM disponible")
    print("  3. Temps estime : ~15-30 min (10 prompts x G=4, 3 epochs)")
    print()
    print("Resultats attendus (RTX 3070, 10 prompts, 3 epochs) :")
    print("  - Loss initial : ~variable (depend des generations)")
    print("  - Reward moyen : augmentation progressive (2.0 -> 3.5)")
    print("  - Qualitative : completions plus structurees, plus de mots-cles")

Skip : entrainement GRPO non execute (LOAD_MODEL_AND_TRAIN=False ou pas de CUDA)

Pour executer :
  1. LOAD_MODEL_AND_TRAIN = True
  2. GPU avec >= 6 Go VRAM disponible
  3. Temps estime : ~15-30 min (10 prompts x G=4, 3 epochs)

Resultats attendus (RTX 3070, 10 prompts, 3 epochs) :
  - Loss initial : ~variable (depend des generations)
  - Reward moyen : augmentation progressive (2.0 -> 3.5)
  - Qualitative : completions plus structurees, plus de mots-cles


## 10. Evaluation qualitative — outputs avant/apres

La metrique cle du GRPO est **behaviorale** : est-ce que les completions deviennent meilleures qualitativement ?

**Methode** :
1. Generer 3 completions pour le meme prompt avant/apres GRPO
2. Comparer la structure, la presence de raisonnement, et le reward moyen
3. Documenter les changements observables

**Pas de claim BEATS quantitative** (terrain pedagogique, 10 prompts = sample insuffisant pour stats). Verdict honnete : "amelioration qualitative observee" ou "INCONCLUSIF".

In [8]:
# Simulation qualitative pedagogique
print("Evaluation qualitative (resultats attendus avec entrainement reel)")
print("=" * 60)

test_prompt = "What is 15 + 27? Think step by step."
print(f"Prompt : {test_prompt}")
print()

print("--- Avant GRPO (modele SFT de base) ---")
print("Completion 1 : 15 + 27 = 42.")
print("Completion 2 : The answer is 42.")
print("Completion 3 : Let me add 15 and 27. The sum is 42.")
print("Reward moyen : ~0.6 (courtes, peu de mots-cles reasoning)")
print()

print("--- Apres GRPO (3 epochs, reward=combined) ---")
print("Completion 1 : Let me think about this. Therefore, 15 + 27 = 42.")
print("Completion 2 : To solve 15 + 27, I add because they are both positive. Hence 42.")
print("Completion 3 : Since 15 + 20 = 35 and 35 + 7 = 42, thus the answer is 42.")
print("Reward moyen : ~0.85 (plus longues, mots-cles reasoning presents)")
print()

print("Changement observable : completions plus longues, plus de mots-cles")
print("de raisonnement (therefore, because, since, hence, thus).")
print()
print("Note : amelioration qualitative, pas de claim BEATS quantitatif.")
print("Sample (10 prompts) insuffisant pour multi-seed >= 4 + 2 sigma.")
print("Verdict : INCONCLUSIF sur metriques, POSITIF qualitativement.")

Evaluation qualitative (resultats attendus avec entrainement reel)
Prompt : What is 15 + 27? Think step by step.

--- Avant GRPO (modele SFT de base) ---
Completion 1 : 15 + 27 = 42.
Completion 2 : The answer is 42.
Completion 3 : Let me add 15 and 27. The sum is 42.
Reward moyen : ~0.6 (courtes, peu de mots-cles reasoning)

--- Apres GRPO (3 epochs, reward=combined) ---
Completion 1 : Let me think about this. Therefore, 15 + 27 = 42.
Completion 2 : To solve 15 + 27, I add because they are both positive. Hence 42.
Completion 3 : Since 15 + 20 = 35 and 35 + 7 = 42, thus the answer is 42.
Reward moyen : ~0.85 (plus longues, mots-cles reasoning presents)

Changement observable : completions plus longues, plus de mots-cles
de raisonnement (therefore, because, since, hence, thus).

Note : amelioration qualitative, pas de claim BEATS quantitatif.
Sample (10 prompts) insuffisant pour multi-seed >= 4 + 2 sigma.
Verdict : INCONCLUSIF sur metriques, POSITIF qualitativement.


## 11. 5 pieges du GRPO en pratique

### Piege 1 : Reward hacking
Le modele peut trouver des shortcuts pour maximiser la reward sans ameliorer reellement la qualite. Ex : repeter "therefore" 50 fois pour maximiser `keyword_reward`.
**Solution** : plafonder les rewards, combiner plusieurs signaux, monitoring qualitatif.

### Piege 2 : KL collapse (beta trop faible)
Si $\beta$ est trop petit, la policy diverge de la reference et genere du texte incoherent.
**Solution** : $\beta = 0.04$ (defaut Deepseek), monitorer KL divergence pendant training.

### Piege 3 : Group size mal calibre
- G trop petit (2) : variance elevee, signal bruite
- G trop grand (16+) : OOM sur GPU 8 Go
**Solution** : G=4 sur RTX 3070, G=8 sur GPU 24 Go. Profiler VRAM avant.

### Piege 4 : Mode collapse
Toutes les completions deviennent identiques (le modele exploite un unique pattern recompense).
**Solution** : temperature > 0 pendant generation, diversifier les reward functions.

### Piege 5 : Reward sparsity
Si la reward est binaire (0 ou 1), l'avantage intra-group est extreme (+1 ou -1) et le gradient est instable.
**Solution** : utiliser des rewards continus (score de similarite, log-probabilite,BLEU partiel).

### Exercice 3 : Implementation de la normalisation intra-group

Le coeur du GRPO est le calcul de l'avantage par normalisation intra-group. L'objectif est d'implementer manuellement cette etape : pour un groupe de G completions avec leurs rewards, calculer les avantages normalises.

**Objectif** : ecrire une fonction `calculer_avantages` qui prend une liste de rewards et retourne les avantages normalises selon la formule GRPO.

**Indices** :
- # Etape 1 : Calculer la moyenne et l'ecart-type du groupe
- # Etape 2 : Appliquer la normalisation : (reward - moyenne) / (ecart_type + epsilon)
- # Indice : ajouter un petit epsilon (1e-8) pour eviter la division par zero quand tous les rewards sont egaux

In [1]:
import numpy as np

def calculer_avantages(rewards: list[float], epsilon: float = 1e-8) -> list[float]:
    # TODO etudiant : implementer la normalisation intra-group GRPO
    
    rewards_array = np.array(rewards)
    
    # Etape 1 : calculer moyenne et ecart-type
    mu = None   # TODO etudiant : np.mean(rewards_array)
    sigma = None  # TODO etudiant : np.std(rewards_array)
    
    # Etape 2 : normaliser
    advantages = None  # TODO etudiant : (rewards_array - mu) / (sigma + epsilon)
    
    return advantages  # TODO etudiant : retourner la liste des avantages

print("Exercice a completer : normalisation intra-group")

Exercice a completer


---

## Bilan — GRPO : la recette Deepseek-R1 (RL pur, sans donnees humaines)

Le GRPO (Group Relative Policy Optimization) est la methode qui a fait emergent
**Deepseek-R1** en janvier 2025 : un modele de raisonnement rivalisant OpenAI o1
**sans donnees de raisonnement humain**. La hierarchie du post-training devient
clair ici : SFT (PT-02) apprend le format, DPO (PT-03) apprend les preferences
humaines, **GRPO apprend a raisonner** via RL pur sur rewards heuristiques.

### L'innovation centrale : l'avantage relatif intra-group

Pour chaque prompt $x$, GRPO echantillonne **G completions** depuis la policy.
Plutot que d'estimer une value function (coûteuse, instable comme en PPO),
GRPO calcule l'avantage par **normalisation intra-group** :

$$\hat{A}_i = \frac{R_i - \mu_G}{\sigma_G}$$

ou $\mu_G$ et $\sigma_G$ sont la moyenne et l'ecart-type des rewards du groupe.
**Pas de critic, pas de value network** : le groupe lui-meme sert de baseline.
C'est cette simplification qui rend le GRPO tractable sur un seul GPU pedagogique.

### Pourquoi GRPO fait emerger du raisonnement (et pas DPO)

- **DPO** consomme des **paires de preferences humaines** statiques — il ne peut
  pas depasser ce que les humains ont montre.
- **GRPO** genere ses propres completions et les **selectionne** par le reward :
  si raisonner en etapes augmente le reward moyen, le gradient pousse vers ce
  comportement — meme si personne ne lui a jamais montre de chain-of-thought.

C'est l'origine de l'**emergence** : le raisonnement n'est pas enseigne, il est
**selectionne** par la structure de la recompense.

### Les 5 pieges du GRPO en pratique

1. **Reward hacking** : le modele trouve des shortcuts (repeter "therefore").
   Solution : plafonner les rewards, combiner plusieurs signaux, monitoring qualitatif.
2. **KL collapse** ($\beta$ trop faible) : la policy diverge de la reference.
3. **Reward sparse** : si aucune completion du groupe ne reussit, avantage = 0 partout.
4. **Groupe trop petit** : $G < 4$ rend la normalisation intra-group trop bruitee.
5. **Reward non-differentiable mal wrappee** : la reward function doit eter uni-variable
   scalaire, sinon le gradient casse.

### Honnetete de l'evaluation (rappel CoursIA)

Ce notebook produit un **proof-of-concept** : single seed, evaluation
**behaviorale qualitative** (avant/apres GRPO), pas de claim BEATS quantitatif.
Toute claim quantitative exige >= 4 seeds + edge >= 2 sigma cross-seed (PT-06).

### Position dans le track GenAI/PostTraining

PT-01 (intro) > PT-02 (SFT) > PT-03 (DPO) > **PT-04 (GRPO, RL heuristique)** >
PT-05 (RLVR, RL verifiable). PT-04 installe le mecanisme GRPO (intra-group
advantage) ; PT-05 va plus loin en remplacant la reward heuristique
**gamable** (longueur, mots-cles) par un **verifieur exact** SymPy — eliminant
par construction le piege n°1 (reward hacking).


## 12. Transition vers PT-05 — RLVR (Reinforcement Learning with Verifiable Rewards)

Le GRPO avec reward function "soft" (longueur, mots-cles) est un bon point de depart. Mais le vrai potentiel du GRPO emerge avec des **rewards verifiables exacts** :

| Aspect | GRPO (ce notebook) | RLVR (PT-05) |
|--------|-------------------|--------------|
| Reward | Fonction heuristique (longueur, mots-cles) | Verifier exact (sympy, exec sandbox) |
| Domaine | General | Mathematique / Code |
| Ambiguite | Possible (reward subjective) | Nulle (correct/incorrect exact) |
| Modele cible | Qwen2.5-0.5B-Instruct | Qwen2.5-Math-1.5B |
| Pattern Deepseek-R1 | Etape 1 (alignment general) | Etape 2 (reasoning mathematique) |

Le RLVR = la variante GRPO qui a fait emerger le **reasoning spontane** dans Deepseek-R1. Quand la recompense est exacte (pas de bruit), le RL explore librement et decouvre des chaines de raisonnement non prevues.

**Prochaines etapes** :
- PT-05 : GRPO + verifier sympy sur GSM8K (math word problems)
- PT-06 : Evaluation comparative SFT/DPO/GRPO/RLVR